# MolDA Dataset — Pipeline Stage Tracker

목적: `raw_v1_10x_rephrase` 최종 데이터셋이 만들어지기까지의 각 pipeline 단계에서
task × split 별 샘플 수가 어떻게 변했는지 (얼마나 탈락했는지) 추적한다.

## Pipeline Stages

| Stage | 위치 | 설명 |
|-------|------|------|
| **A. Source** | HF Hub / 로컬 CSV | 원본 (SMolInstruct, ChEBI-20-MM, Mol-Instructions, DeepChem, BACE) |
| **B. Step 1** | `Raw/raw_v1/step1/{task}_subtask-0_{split}/` | SMILES→graph+SELFIES 변환 실패 drop. Mol-Instr 98/2 train/val split. name_conv val/test skip |
| **C. Step 2** | `Raw/raw_v1/step2/{task}_subtask-0_{split}/` | Cross-source decontamination (eval blacklist + within-family dedup) |
| **D. Processed raw_v1** | `Processed/raw_v1/{Train,Val,Test}/` | split concat + prompt 포매팅 + SELFIES strict 재필터 |
| **E. Processed raw_v1_10x_rephrase** | `Processed/raw_v1_10x_rephrase/{Train,Val,Test}/` | 4 target task × 10× instruction rephrase (Train만) |

※ Raw/step1·step2는 `raw_v1` tag에만 존재 (10x_rephrase는 Processed 단계부터 분기).


## Section 0 — Setup & Helpers


In [1]:
import os
import re
import sys
import warnings
from collections import defaultdict, Counter

import pandas as pd
import pyarrow.ipc as ipc

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = "/opt/EMNLP_MolDA/New_MolDA"
RAW_DATA_ROOT = os.path.join(PROJECT_ROOT, "dataset/Raw")
PROCESSED_DATA_ROOT = os.path.join(PROJECT_ROOT, "dataset/Processed")

# 핵심: Raw/step1, step2는 BASE_TAG에만 존재. 10x_rephrase는 Processed 단계.
BASE_TAG = "raw_v1"
AUG_TAG = "raw_v1_10x_rephrase"

STEP1_ROOT = os.path.join(RAW_DATA_ROOT, BASE_TAG, "step1")
STEP2_ROOT = os.path.join(RAW_DATA_ROOT, BASE_TAG, "step2")
PROC_BASE_ROOT = os.path.join(PROCESSED_DATA_ROOT, BASE_TAG)
PROC_AUG_ROOT = os.path.join(PROCESSED_DATA_ROOT, AUG_TAG)

for name, path in [
    ("Step 1", STEP1_ROOT), ("Step 2", STEP2_ROOT),
    ("Processed (base)", PROC_BASE_ROOT), ("Processed (10x)", PROC_AUG_ROOT),
]:
    print(f"  {name:20s} {path}  exists={os.path.isdir(path)}")

sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))


  Step 1               /opt/EMNLP_MolDA/New_MolDA/dataset/Raw/raw_v1/step1  exists=True
  Step 2               /opt/EMNLP_MolDA/New_MolDA/dataset/Raw/raw_v1/step2  exists=True
  Processed (base)     /opt/EMNLP_MolDA/New_MolDA/dataset/Processed/raw_v1  exists=True
  Processed (10x)      /opt/EMNLP_MolDA/New_MolDA/dataset/Processed/raw_v1_10x_rephrase  exists=True


In [2]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def count_arrow_rows(dataset_dir):
    """Arrow shard 디렉터리의 총 row 수 반환. 없으면 None."""
    if not os.path.isdir(dataset_dir):
        return None
    arrow_files = sorted(
        f for f in os.listdir(dataset_dir)
        if f.startswith("data-") and f.endswith(".arrow")
    )
    if not arrow_files:
        return 0
    total = 0
    for af in arrow_files:
        path = os.path.join(dataset_dir, af)
        try:
            reader = ipc.open_stream(path)
            for batch in reader:
                total += batch.num_rows
        except Exception:
            try:
                table = ipc.open_file(path).read_all()
                total += len(table)
            except Exception:
                pass
    return total


_TASK_SUBTASK_RE = re.compile(
    r"^(.+)_subtask-(\d+|multi_label_classification)_(train|val|test)$"
)


def scan_per_task_counts(per_task_root):
    """{task}_subtask-{idx}_{split} 디렉터리들을 스캔해 task→split→count 맵 반환."""
    results = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})
    if not os.path.isdir(per_task_root):
        return {}
    for entry in sorted(os.listdir(per_task_root)):
        m = _TASK_SUBTASK_RE.match(entry)
        if not m:
            continue
        task_name, _subtask, split = m.groups()
        count = count_arrow_rows(os.path.join(per_task_root, entry))
        if count is not None:
            # 같은 task에 multi-subtask가 있으면 합산
            results[task_name][split] = results[task_name].get(split, 0) + count
    return dict(results)


def count_by_task_in_concat(split_dir):
    """Processed/ concat dataset을 열어 task별 row 수를 Counter로 반환."""
    from datasets import load_from_disk
    if not os.path.isdir(split_dir):
        return {}
    ds = load_from_disk(split_dir)
    return dict(Counter(ds["task"]))


print("Helpers ready.")


Helpers ready.


In [3]:
# ---------------------------------------------------------------------------
# Task taxonomy (generator.py 기반)
# ---------------------------------------------------------------------------
SOURCE_TASK_MAP = {
    "SMolInstruct": [
        "smol-forward_synthesis", "smol-retrosynthesis",
        "smol-molecule_captioning", "smol-molecule_generation",
        "smol-property_prediction-bbbp", "smol-property_prediction-clintox",
        "smol-property_prediction-esol", "smol-property_prediction-hiv",
        "smol-property_prediction-lipo", "smol-property_prediction-sider",
        "smol-name_conversion-i2s", "smol-name_conversion-s2i",
    ],
    "Mol-Instruction": [
        "forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
        "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap",
    ],
    "ChEBI-20": [
        "chebi-20-mol2text", "chebi-20-text2mol",
    ],
}
TASK_TO_SOURCE = {t: s for s, tasks in SOURCE_TASK_MAP.items() for t in tasks}

# 최종 표의 Task group
TASK_GROUP = {
    # Regression
    "qm9_homo": "Property Prediction (Regression)",
    "qm9_lumo": "Property Prediction (Regression)",
    "qm9_homo_lumo_gap": "Property Prediction (Regression)",
    "smol-property_prediction-esol": "Property Prediction (Regression)",
    "smol-property_prediction-lipo": "Property Prediction (Regression)",
    # Classification
    "smol-property_prediction-bbbp": "Property Prediction (Classification)",
    "smol-property_prediction-clintox": "Property Prediction (Classification)",
    "smol-property_prediction-hiv": "Property Prediction (Classification)",
    "smol-property_prediction-sider": "Property Prediction (Classification)",
    # Reaction
    "forward_reaction_prediction": "Forward Reaction Prediction",
    "smol-forward_synthesis": "Forward Reaction Prediction",
    "retrosynthesis": "Retrosynthesis",
    "smol-retrosynthesis": "Retrosynthesis",
    "reagent_prediction": "Reagent Prediction",
    # Mol<->Text
    "chebi-20-mol2text": "Molecule Captioning",
    "smol-molecule_captioning": "Molecule Captioning",
    "chebi-20-text2mol": "Description-Guided Molecule Generation",
    "smol-molecule_generation": "Description-Guided Molecule Generation",
    # Name conversion
    "smol-name_conversion-i2s": "Name Conversion",
    "smol-name_conversion-s2i": "Name Conversion",
}

ALL_TASKS = sorted(TASK_GROUP.keys())
print(f"Total tasks in scope: {len(ALL_TASKS)}")


Total tasks in scope: 20


## Section 1 — Stage A: Source Raw Counts

HuggingFace 원본 데이터셋을 로드해 **원본 (Stage A)** 에서의 task × split 샘플 수를
집계. SMolInstruct와 ChEBI-20-MM은 split이 shipped 되어 있고, Mol-Instructions는
metadata 필드에서 train/test를 구분 (val은 generator.py가 2% 잘라 만듦 → Stage A에는
없음).

처음 실행 시 HF 캐시가 없으면 수분 소요 가능. 이미 캐시되어 있으면 빠름.
`LOAD_FROM_SOURCE=False`로 두면 skip하고 known reference 값만 사용.


In [4]:
LOAD_FROM_SOURCE = True

raw_source_counts = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})

if LOAD_FROM_SOURCE:
    from datasets import load_dataset

    # --- SMolInstruct -----------------------------------------------------
    print("[Loading] osunlp/SMolInstruct...")
    smol = load_dataset(
        "osunlp/SMolInstruct",
        use_selfies=False, insert_core_tags=False, trust_remote_code=True,
    )
    for split_name, split_ds in smol.items():
        # split_name: train / validation / test
        key = {"train": "train", "validation": "val", "test": "test"}.get(split_name)
        if key is None:
            continue
        c = Counter(split_ds["task"])
        for t, n in c.items():
            raw_source_counts[f"smol-{t}"][key] = n
    print(f"  SMolInstruct splits: {list(smol.keys())} "
          f"(train={len(smol['train']):,}, val={len(smol.get('validation', [])):,}, test={len(smol['test']):,})")

    # --- ChEBI-20-MM ------------------------------------------------------
    print("[Loading] liupf/ChEBI-20-MM...")
    chebi = load_dataset("liupf/ChEBI-20-MM")
    for task in ["chebi-20-mol2text", "chebi-20-text2mol"]:
        raw_source_counts[task] = {
            "train": len(chebi["train"]),
            "val": len(chebi["validation"]),
            "test": len(chebi["test"]),
        }
    print(f"  ChEBI-20-MM: train={len(chebi['train']):,}, "
          f"val={len(chebi['validation']):,}, test={len(chebi['test']):,}")

    # --- Mol-Instructions -------------------------------------------------
    print("[Loading] zjunlp/Mol-Instructions (Molecule-oriented)...")
    from dataset_generation import instructions_smol
    mol_inst = load_dataset(
        "zjunlp/Mol-Instructions", "Molecule-oriented Instructions",
        trust_remote_code=True,
    )
    # reaction / retro / reagent: metadata에 train/test 태그
    for task in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
        ds = mol_inst[task]
        meta = ds["metadata"]
        n_train = sum(1 for m in meta if "train" in m)
        n_test = sum(1 for m in meta if "test" in m)
        raw_source_counts[task] = {"train": n_train, "val": 0, "test": n_test}
    # qm9_homo/lumo/gap: property_prediction에서 instruction 기준 필터
    pp = mol_inst["property_prediction"]
    for task in ["qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap"]:
        subtask = task.replace("qm9_", "")
        templates = getattr(instructions_smol, f"filtering_template_{subtask}")
        templates_set = set(templates)
        matches = [i for i, ins in enumerate(pp["instruction"]) if ins in templates_set]
        meta = [pp["metadata"][i] for i in matches]
        n_train = sum(1 for m in meta if "train" in m)
        n_test = sum(1 for m in meta if "test" in m)
        raw_source_counts[task] = {"train": n_train, "val": 0, "test": n_test}
    print(f"  Mol-Instructions: "
          f"forward={len(mol_inst['forward_reaction_prediction']):,}, "
          f"retro={len(mol_inst['retrosynthesis']):,}, "
          f"reagent={len(mol_inst['reagent_prediction']):,}, "
          f"property={len(pp):,}")
else:
    print("[Skip] LOAD_FROM_SOURCE=False")

raw_source_counts = dict(raw_source_counts)
print(f"\n=== Stage A (Source Raw) — tasks={len(raw_source_counts)} ===")
df_A = pd.DataFrame([
    {"Task": t, "Source": TASK_TO_SOURCE.get(t, "?"),
     "A_train": v["train"], "A_val": v["val"], "A_test": v["test"]}
    for t, v in sorted(raw_source_counts.items())
])
display(df_A.style.format({"A_train": "{:,}", "A_val": "{:,}", "A_test": "{:,}"}))


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'osunlp/SMolInstruct' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


[Loading] osunlp/SMolInstruct...


Using the latest cached version of the dataset since osunlp/SMolInstruct couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default-34348c3c9b2317bd' at /root/.cache/huggingface/datasets/osunlp___s_mol_instruct/default-34348c3c9b2317bd/1.3.0/9a601a046068582d752b0e8b639db0845f457fb75d210af6faa2f51afed62cd2 (last modified on Wed Apr 22 08:50:30 2026).


  SMolInstruct splits: ['train', 'validation', 'test'] (train=3,288,855, val=20,498, test=33,061)
[Loading] liupf/ChEBI-20-MM...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'zjunlp/Mol-Instructions' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ChEBI-20-MM: train=26,402, val=3,299, test=3,297
[Loading] zjunlp/Mol-Instructions (Molecule-oriented)...


Using the latest cached version of the dataset since zjunlp/Mol-Instructions couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'Molecule-oriented Instructions' at /root/.cache/huggingface/datasets/zjunlp___mol-instructions/Molecule-oriented Instructions/1.0.0/1d5207d9a604ed411050d386c45222df527905f70e04e090155aa95beeb81eb8 (last modified on Wed Apr 22 08:47:18 2026).


  Mol-Instructions: forward=125,384, retro=129,684, reagent=125,384, property=362,100

=== Stage A (Source Raw) — tasks=22 ===


,Task,Source,A_train,A_val,A_test
0,chebi-20-mol2text,ChEBI-20,"26,402","3,299","3,297"
1,chebi-20-text2mol,ChEBI-20,"26,402","3,299","3,297"
2,forward_reaction_prediction,Mol-Instruction,"124,384",0,"1,000"
3,qm9_homo,Mol-Instruction,"120,062",0,684
4,qm9_homo_lumo_gap,Mol-Instruction,"119,940",0,661
5,qm9_lumo,Mol-Instruction,"120,111",0,642
6,reagent_prediction,Mol-Instruction,"124,384",0,"1,000"
7,retrosynthesis,Mol-Instruction,"128,684",0,"1,000"
8,smol-forward_synthesis,SMolInstruct,"971,809","2,049","4,062"
9,smol-molecule_captioning,SMolInstruct,"56,498","1,269","2,538"


## Section 2 — Stage B (Step 1) & Stage C (Step 2)

`Raw/raw_v1/step1/` 와 `Raw/raw_v1/step2/` per-task arrow 디렉터리를 스캔.


In [5]:
step1 = scan_per_task_counts(STEP1_ROOT)
step2 = scan_per_task_counts(STEP2_ROOT)

print(f"Step 1: {len(step1)} tasks")
print(f"Step 2: {len(step2)} tasks")

rows = []
for t in ALL_TASKS:
    b = step1.get(t, {"train": 0, "val": 0, "test": 0})
    c = step2.get(t, {"train": 0, "val": 0, "test": 0})
    rows.append({
        "Task": t, "Source": TASK_TO_SOURCE.get(t, "?"),
        "B_train": b["train"], "B_val": b["val"], "B_test": b["test"],
        "C_train": c["train"], "C_val": c["val"], "C_test": c["test"],
    })
df_BC = pd.DataFrame(rows)
display(df_BC.style.format({col: "{:,}" for col in df_BC.columns if col.startswith(("B_", "C_"))}))


Step 1: 21 tasks
Step 2: 21 tasks


,Task,Source,B_train,B_val,B_test,C_train,C_val,C_test
0,chebi-20-mol2text,ChEBI-20,"26,402","3,299","3,297","25,908","3,299","3,297"
1,chebi-20-text2mol,ChEBI-20,"26,402","3,299","3,297","25,908","3,299","3,297"
2,forward_reaction_prediction,Mol-Instruction,"121,896","2,488","1,000","28,107","2,488","1,000"
3,qm9_homo,Mol-Instruction,"117,660","2,402",684,"117,656","2,402",684
4,qm9_homo_lumo_gap,Mol-Instruction,"117,541","2,399",661,"117,536","2,399",661
5,qm9_lumo,Mol-Instruction,"117,708","2,403",642,"117,706","2,403",642
6,reagent_prediction,Mol-Instruction,"121,896","2,488","1,000","121,896","2,488","1,000"
7,retrosynthesis,Mol-Instruction,"126,110","2,574","1,000","2,732","2,574","1,000"
8,smol-forward_synthesis,SMolInstruct,"971,766","2,049","4,062","969,057","2,049","4,062"
9,smol-molecule_captioning,SMolInstruct,"56,497","1,269","2,538","26,407","1,269","2,538"


## Section 3 — Stage D: Processed/raw_v1 (concat + SELFIES filter)


In [6]:
D_counts = {t: {"train": 0, "val": 0, "test": 0} for t in ALL_TASKS}
for split_name, sub in [("train", "Train"), ("val", "Val"), ("test", "Test")]:
    by_task = count_by_task_in_concat(os.path.join(PROC_BASE_ROOT, sub))
    for t, n in by_task.items():
        if t in D_counts:
            D_counts[t][split_name] = n

rows = []
for t in ALL_TASKS:
    v = D_counts[t]
    rows.append({
        "Task": t, "Source": TASK_TO_SOURCE.get(t, "?"),
        "D_train": v["train"], "D_val": v["val"], "D_test": v["test"],
    })
df_D = pd.DataFrame(rows)
display(df_D.style.format({"D_train": "{:,}", "D_val": "{:,}", "D_test": "{:,}"}))

print(f"\nProcessed/raw_v1 totals: "
      f"Train={df_D['D_train'].sum():,}, "
      f"Val={df_D['D_val'].sum():,}, "
      f"Test={df_D['D_test'].sum():,}")


,Task,Source,D_train,D_val,D_test
0,chebi-20-mol2text,ChEBI-20,"25,908","3,299","3,297"
1,chebi-20-text2mol,ChEBI-20,"25,908","3,299","3,297"
2,forward_reaction_prediction,Mol-Instruction,"28,104","2,488","1,000"
3,qm9_homo,Mol-Instruction,"117,656","2,402",684
4,qm9_homo_lumo_gap,Mol-Instruction,"117,536","2,399",661
5,qm9_lumo,Mol-Instruction,"117,706","2,403",642
6,reagent_prediction,Mol-Instruction,"121,882","2,488","1,000"
7,retrosynthesis,Mol-Instruction,"2,732","2,574","1,000"
8,smol-forward_synthesis,SMolInstruct,"968,849","2,049","4,060"
9,smol-molecule_captioning,SMolInstruct,"26,403","1,269","2,538"



Processed/raw_v1 totals: Train=3,177,891, Val=35,865, Test=32,665


## Section 4 — Stage E: Processed/raw_v1_10x_rephrase (final)

4개 target task(`chebi-20-mol2text`, `chebi-20-text2mol`, `smol-molecule_captioning`,
`smol-molecule_generation`)의 Train만 10× 증강. Val/Test 및 나머지 17 task는 D와 동일.


In [7]:
E_counts = {t: {"train": 0, "val": 0, "test": 0} for t in ALL_TASKS}
for split_name, sub in [("train", "Train"), ("val", "Val"), ("test", "Test")]:
    by_task = count_by_task_in_concat(os.path.join(PROC_AUG_ROOT, sub))
    for t, n in by_task.items():
        if t in E_counts:
            E_counts[t][split_name] = n

rows = []
for t in ALL_TASKS:
    v = E_counts[t]
    rows.append({
        "Task": t, "Source": TASK_TO_SOURCE.get(t, "?"),
        "E_train": v["train"], "E_val": v["val"], "E_test": v["test"],
    })
df_E = pd.DataFrame(rows)
display(df_E.style.format({"E_train": "{:,}", "E_val": "{:,}", "E_test": "{:,}"}))

print(f"\nProcessed/raw_v1_10x_rephrase totals: "
      f"Train={df_E['E_train'].sum():,}, "
      f"Val={df_E['E_val'].sum():,}, "
      f"Test={df_E['E_test'].sum():,}")

# 10x expansion 검증
TARGET_10X = ["chebi-20-mol2text", "chebi-20-text2mol",
              "smol-molecule_captioning", "smol-molecule_generation"]
print("\n=== 10x expansion check (Train only) ===")
for t in TARGET_10X:
    d = D_counts[t]["train"]
    e = E_counts[t]["train"]
    ratio = e / d if d else float("nan")
    print(f"  {t:35s}  D_train={d:>9,}  E_train={e:>9,}  ratio={ratio:.2f}x")


,Task,Source,E_train,E_val,E_test
0,chebi-20-mol2text,ChEBI-20,"259,080","3,299","3,297"
1,chebi-20-text2mol,ChEBI-20,"259,080","3,299","3,297"
2,forward_reaction_prediction,Mol-Instruction,"28,104","2,488","1,000"
3,qm9_homo,Mol-Instruction,"117,656","2,402",684
4,qm9_homo_lumo_gap,Mol-Instruction,"117,536","2,399",661
5,qm9_lumo,Mol-Instruction,"117,706","2,403",642
6,reagent_prediction,Mol-Instruction,"121,882","2,488","1,000"
7,retrosynthesis,Mol-Instruction,"2,732","2,574","1,000"
8,smol-forward_synthesis,SMolInstruct,"968,849","2,049","4,060"
9,smol-molecule_captioning,SMolInstruct,"264,030","1,269","2,538"



Processed/raw_v1_10x_rephrase totals: Train=4,119,480, Val=35,865, Test=32,665

=== 10x expansion check (Train only) ===
  chebi-20-mol2text                    D_train=   25,908  E_train=  259,080  ratio=10.00x
  chebi-20-text2mol                    D_train=   25,908  E_train=  259,080  ratio=10.00x
  smol-molecule_captioning             D_train=   26,403  E_train=  264,030  ratio=10.00x
  smol-molecule_generation             D_train=   26,402  E_train=  264,020  ratio=10.00x


## Section 5 — Master Table (Task × Stage × Split)

Stage A→B→C→D→E 전체 흐름과 각 전이에서의 drop을 한 DataFrame으로 정리.


In [8]:
def _get(d, t, k):
    return d.get(t, {}).get(k, 0)

master_rows = []
for t in ALL_TASKS:
    row = {"Task": t, "Source": TASK_TO_SOURCE.get(t, "?"),
           "Group": TASK_GROUP.get(t, "?")}
    for split in ["train", "val", "test"]:
        a = _get(raw_source_counts, t, split)
        b = _get(step1, t, split)
        c = _get(step2, t, split)
        d = D_counts[t][split]
        e = E_counts[t][split]
        row[f"A_{split}"] = a
        row[f"B_{split}"] = b
        row[f"C_{split}"] = c
        row[f"D_{split}"] = d
        row[f"E_{split}"] = e
        row[f"dropAB_{split}"] = (a - b) if a else 0
        row[f"dropBC_{split}"] = (b - c) if b else 0
        row[f"dropCD_{split}"] = (c - d) if c else 0
        row[f"deltaDE_{split}"] = e - d  # 증가이면 양수
    master_rows.append(row)

df_master = pd.DataFrame(master_rows)

# 보기 편한 열 순서: 각 split별로 A, B, dropAB, C, dropBC, D, dropCD, E, deltaDE
ordered_cols = ["Task", "Group", "Source"]
for split in ["train", "val", "test"]:
    ordered_cols += [f"A_{split}", f"B_{split}", f"dropAB_{split}",
                     f"C_{split}", f"dropBC_{split}",
                     f"D_{split}", f"dropCD_{split}",
                     f"E_{split}", f"deltaDE_{split}"]
df_master = df_master[ordered_cols]
print(f"Master table: {df_master.shape}")
df_master.head(3)


Master table: (20, 30)


,Task,Group,Source,A_train,B_train,dropAB_train,C_train,dropBC_train,D_train,dropCD_train,...,deltaDE_val,A_test,B_test,dropAB_test,C_test,dropBC_test,D_test,dropCD_test,E_test,deltaDE_test
0,chebi-20-mol2text,Molecule Captioning,ChEBI-20,26402,26402,0,25908,494,25908,0,...,0,3297,3297,0,3297,0,3297,0,3297,0
1,chebi-20-text2mol,Description-Guided Molecule Generation,ChEBI-20,26402,26402,0,25908,494,25908,0,...,0,3297,3297,0,3297,0,3297,0,3297,0
2,forward_reaction_prediction,Forward Reaction Prediction,Mol-Instruction,124384,121896,2488,28107,93789,28104,3,...,0,1000,1000,0,1000,0,1000,0,1000,0


In [9]:
# Split별 separate view (보기 편함)
def view_by_split(split):
    cols = ["Task", "Group", "Source",
            f"A_{split}", f"B_{split}", f"dropAB_{split}",
            f"C_{split}", f"dropBC_{split}",
            f"D_{split}", f"dropCD_{split}",
            f"E_{split}", f"deltaDE_{split}"]
    sub = df_master[cols].copy()
    sub.columns = ["Task", "Group", "Source",
                   "A", "B", "drop A→B",
                   "C", "drop B→C", "D", "drop C→D",
                   "E", "delta D→E"]
    return sub.sort_values(["Group", "Source", "Task"]).reset_index(drop=True)

print("\n=== TRAIN split ===")
display(view_by_split("train").style.format(
    {c: "{:,}" for c in ["A", "B", "drop A→B", "C", "drop B→C",
                          "D", "drop C→D", "E", "delta D→E"]}))



=== TRAIN split ===


,Task,Group,Source,A,B,drop A→B,C,drop B→C,D,drop C→D,E,delta D→E
0,chebi-20-text2mol,Description-Guided Molecule Generation,ChEBI-20,"26,402","26,402",0,"25,908",494,"25,908",0,"259,080","233,172"
1,smol-molecule_generation,Description-Guided Molecule Generation,SMolInstruct,"56,498","56,498",0,"26,407","30,091","26,402",5,"264,020","237,618"
2,forward_reaction_prediction,Forward Reaction Prediction,Mol-Instruction,"124,384","121,896","2,488","28,107","93,789","28,104",3,"28,104",0
3,smol-forward_synthesis,Forward Reaction Prediction,SMolInstruct,"971,809","971,766",43,"969,057","2,709","968,849",208,"968,849",0
4,chebi-20-mol2text,Molecule Captioning,ChEBI-20,"26,402","26,402",0,"25,908",494,"25,908",0,"259,080","233,172"
5,smol-molecule_captioning,Molecule Captioning,SMolInstruct,"56,498","56,497",1,"26,407","30,090","26,403",4,"264,030","237,627"
6,smol-name_conversion-i2s,Name Conversion,SMolInstruct,"299,890","299,890",0,"299,890",0,"299,247",643,"299,247",0
7,smol-name_conversion-s2i,Name Conversion,SMolInstruct,"299,890","299,890",0,"299,890",0,"299,247",643,"299,247",0
8,smol-property_prediction-bbbp,Property Prediction (Classification),SMolInstruct,"1,569","1,569",0,"1,569",0,"1,569",0,"1,569",0
9,smol-property_prediction-clintox,Property Prediction (Classification),SMolInstruct,"1,144","1,144",0,"1,144",0,"1,144",0,"1,144",0


In [10]:
print("=== VAL split ===")
display(view_by_split("val").style.format(
    {c: "{:,}" for c in ["A", "B", "drop A→B", "C", "drop B→C",
                          "D", "drop C→D", "E", "delta D→E"]}))


=== VAL split ===


,Task,Group,Source,A,B,drop A→B,C,drop B→C,D,drop C→D,E,delta D→E
0,chebi-20-text2mol,Description-Guided Molecule Generation,ChEBI-20,"3,299","3,299",0,"3,299",0,"3,299",0,"3,299",0
1,smol-molecule_generation,Description-Guided Molecule Generation,SMolInstruct,"1,269","1,269",0,"1,269",0,"1,269",0,"1,269",0
2,forward_reaction_prediction,Forward Reaction Prediction,Mol-Instruction,0,"2,488",0,"2,488",0,"2,488",0,"2,488",0
3,smol-forward_synthesis,Forward Reaction Prediction,SMolInstruct,"2,049","2,049",0,"2,049",0,"2,049",0,"2,049",0
4,chebi-20-mol2text,Molecule Captioning,ChEBI-20,"3,299","3,299",0,"3,299",0,"3,299",0,"3,299",0
5,smol-molecule_captioning,Molecule Captioning,SMolInstruct,"1,269","1,269",0,"1,269",0,"1,269",0,"1,269",0
6,smol-name_conversion-i2s,Name Conversion,SMolInstruct,"1,496",0,"1,496",0,0,0,0,0,0
7,smol-name_conversion-s2i,Name Conversion,SMolInstruct,"1,496",0,"1,496",0,0,0,0,0,0
8,smol-property_prediction-bbbp,Property Prediction (Classification),SMolInstruct,196,196,0,196,0,196,0,196,0
9,smol-property_prediction-clintox,Property Prediction (Classification),SMolInstruct,143,143,0,143,0,143,0,143,0


In [11]:
print("=== TEST split ===")
display(view_by_split("test").style.format(
    {c: "{:,}" for c in ["A", "B", "drop A→B", "C", "drop B→C",
                          "D", "drop C→D", "E", "delta D→E"]}))


=== TEST split ===


,Task,Group,Source,A,B,drop A→B,C,drop B→C,D,drop C→D,E,delta D→E
0,chebi-20-text2mol,Description-Guided Molecule Generation,ChEBI-20,"3,297","3,297",0,"3,297",0,"3,297",0,"3,297",0
1,smol-molecule_generation,Description-Guided Molecule Generation,SMolInstruct,"2,493","2,493",0,"2,493",0,"2,493",0,"2,493",0
2,forward_reaction_prediction,Forward Reaction Prediction,Mol-Instruction,"1,000","1,000",0,"1,000",0,"1,000",0,"1,000",0
3,smol-forward_synthesis,Forward Reaction Prediction,SMolInstruct,"4,062","4,062",0,"4,062",0,"4,060",2,"4,060",0
4,chebi-20-mol2text,Molecule Captioning,ChEBI-20,"3,297","3,297",0,"3,297",0,"3,297",0,"3,297",0
5,smol-molecule_captioning,Molecule Captioning,SMolInstruct,"2,538","2,538",0,"2,538",0,"2,538",0,"2,538",0
6,smol-name_conversion-i2s,Name Conversion,SMolInstruct,"2,993",0,"2,993",0,0,0,0,0,0
7,smol-name_conversion-s2i,Name Conversion,SMolInstruct,"2,993",0,"2,993",0,0,0,0,0,0
8,smol-property_prediction-bbbp,Property Prediction (Classification),SMolInstruct,197,197,0,197,0,197,0,197,0
9,smol-property_prediction-clintox,Property Prediction (Classification),SMolInstruct,144,144,0,144,0,144,0,144,0


## Section 6 — Group-level Summary

Task group(Property Regression/Classification, Reaction, …) 단위로 Stage별 총합.


In [12]:
GROUP_ORDER = [
    "Property Prediction (Regression)",
    "Property Prediction (Classification)",
    "Forward Reaction Prediction",
    "Retrosynthesis",
    "Reagent Prediction",
    "Molecule Captioning",
    "Description-Guided Molecule Generation",
    "Name Conversion",
]

# group × stage × split 합계
group_rows = []
for group in GROUP_ORDER:
    sub = df_master[df_master["Group"] == group]
    if sub.empty:
        continue
    srcs = sorted(sub["Source"].unique())
    row = {
        "Group": group,
        "Source(s)": " + ".join(srcs),
    }
    for split in ["train", "val", "test"]:
        for stage in ["A", "B", "C", "D", "E"]:
            row[f"{stage}_{split}"] = int(sub[f"{stage}_{split}"].sum())
    group_rows.append(row)

# Overall
overall = {"Group": "Overall", "Source(s)": "—"}
for split in ["train", "val", "test"]:
    for stage in ["A", "B", "C", "D", "E"]:
        overall[f"{stage}_{split}"] = int(df_master[f"{stage}_{split}"].sum())
group_rows.append(overall)

df_group = pd.DataFrame(group_rows)

# Train 흐름만 보기
cols_train = ["Group", "Source(s)", "A_train", "B_train", "C_train", "D_train", "E_train"]
print("=== Group summary: TRAIN flow (Raw → Step1 → Step2 → Processed → 10x) ===")
display(df_group[cols_train].style.format({c: "{:,}" for c in cols_train if c.startswith(("A_", "B_", "C_", "D_", "E_"))}))


=== Group summary: TRAIN flow (Raw → Step1 → Step2 → Processed → 10x) ===


,Group,Source(s),A_train,B_train,C_train,D_train,E_train
0,Property Prediction (Regression),Mol-Instruction + SMolInstruct,"364,361","357,157","357,146","357,146","357,146"
1,Property Prediction (Classification),SMolInstruct,"58,397","58,396","58,396","58,396","58,396"
2,Forward Reaction Prediction,Mol-Instruction + SMolInstruct,"1,096,193","1,093,662","997,164","996,953","996,953"
3,Retrosynthesis,Mol-Instruction + SMolInstruct,"1,070,419","1,067,845","940,424","940,399","940,399"
4,Reagent Prediction,Mol-Instruction,"124,384","121,896","121,896","121,882","121,882"
5,Molecule Captioning,ChEBI-20 + SMolInstruct,"82,900","82,899","52,315","52,311","523,110"
6,Description-Guided Molecule Generation,ChEBI-20 + SMolInstruct,"82,900","82,900","52,315","52,310","523,100"
7,Name Conversion,SMolInstruct,"599,780","599,780","599,780","598,494","598,494"
8,Overall,—,"3,479,334","3,464,535","3,179,436","3,177,891","4,119,480"


In [13]:
# 동일 요약을 Val, Test에도
for split in ["val", "test"]:
    cols = ["Group", "Source(s)"] + [f"{s}_{split}" for s in ["A", "B", "C", "D", "E"]]
    print(f"\n=== Group summary: {split.upper()} flow ===")
    display(df_group[cols].style.format(
        {c: "{:,}" for c in cols if c.startswith(("A_", "B_", "C_", "D_", "E_"))}))



=== Group summary: VAL flow ===


,Group,Source(s),A_val,B_val,C_val,D_val,E_val
0,Property Prediction (Regression),Mol-Instruction + SMolInstruct,531,"7,735","7,735","7,735","7,735"
1,Property Prediction (Classification),SMolInstruct,"7,303","7,303","7,303","7,303","7,303"
2,Forward Reaction Prediction,Mol-Instruction + SMolInstruct,"2,049","4,537","4,537","4,537","4,537"
3,Retrosynthesis,Mol-Instruction + SMolInstruct,"2,092","4,666","4,666","4,666","4,666"
4,Reagent Prediction,Mol-Instruction,0,"2,488","2,488","2,488","2,488"
5,Molecule Captioning,ChEBI-20 + SMolInstruct,"4,568","4,568","4,568","4,568","4,568"
6,Description-Guided Molecule Generation,ChEBI-20 + SMolInstruct,"4,568","4,568","4,568","4,568","4,568"
7,Name Conversion,SMolInstruct,"2,992",0,0,0,0
8,Overall,—,"24,103","35,865","35,865","35,865","35,865"



=== Group summary: TEST flow ===


,Group,Source(s),A_test,B_test,C_test,D_test,E_test
0,Property Prediction (Regression),Mol-Instruction + SMolInstruct,"2,519","2,519","2,519","2,519","2,519"
1,Property Prediction (Classification),SMolInstruct,"7,308","7,307","7,307","7,306","7,306"
2,Forward Reaction Prediction,Mol-Instruction + SMolInstruct,"5,062","5,062","5,062","5,060","5,060"
3,Retrosynthesis,Mol-Instruction + SMolInstruct,"5,156","5,156","5,156","5,155","5,155"
4,Reagent Prediction,Mol-Instruction,"1,000","1,000","1,000","1,000","1,000"
5,Molecule Captioning,ChEBI-20 + SMolInstruct,"5,835","5,835","5,835","5,835","5,835"
6,Description-Guided Molecule Generation,ChEBI-20 + SMolInstruct,"5,790","5,790","5,790","5,790","5,790"
7,Name Conversion,SMolInstruct,"5,986",0,0,0,0
8,Overall,—,"38,656","32,669","32,669","32,665","32,665"


In [14]:
# 드롭/증가만 한 줄로: "A → B (-xx) → C (-yy) → D (-zz) → E (+kk)"
def flow_str(row, split):
    a = row[f"A_{split}"]
    b = row[f"B_{split}"]
    c = row[f"C_{split}"]
    d = row[f"D_{split}"]
    e = row[f"E_{split}"]
    return (f"{a:>9,} → {b:>9,} (Δ{b-a:+,}) → {c:>9,} (Δ{c-b:+,}) "
            f"→ {d:>9,} (Δ{d-c:+,}) → {e:>9,} (Δ{e-d:+,})")

print("=" * 110)
print(f"{'Group':42s} {'Split':5s}  A          → B          → C          → D          → E")
print("=" * 110)
for _, row in df_group.iterrows():
    for split in ["train", "val", "test"]:
        print(f"{row['Group'][:42]:42s} {split:5s}  {flow_str(row, split)}")
    print("-" * 110)


Group                                      Split  A          → B          → C          → D          → E
Property Prediction (Regression)           train    364,361 →   357,157 (Δ-7,204) →   357,146 (Δ-11) →   357,146 (Δ+0) →   357,146 (Δ+0)
Property Prediction (Regression)           val          531 →     7,735 (Δ+7,204) →     7,735 (Δ+0) →     7,735 (Δ+0) →     7,735 (Δ+0)
Property Prediction (Regression)           test       2,519 →     2,519 (Δ+0) →     2,519 (Δ+0) →     2,519 (Δ+0) →     2,519 (Δ+0)
--------------------------------------------------------------------------------------------------------------
Property Prediction (Classification)       train     58,397 →    58,396 (Δ-1) →    58,396 (Δ+0) →    58,396 (Δ+0) →    58,396 (Δ+0)
Property Prediction (Classification)       val        7,303 →     7,303 (Δ+0) →     7,303 (Δ+0) →     7,303 (Δ+0) →     7,303 (Δ+0)
Property Prediction (Classification)       test       7,308 →     7,307 (Δ-1) →     7,307 (Δ+0) →     7,306 (Δ-1) → 

## Section 7 — Per-Task Drop Diagnostics

각 task × split에서 발생한 drop을 코드상의 원인(메커니즘)과 매칭.

원인 태그:
- `NAME_CONV_SKIP`: `smol-name_conversion-*`는 Step 1에서 train만 저장 (val/test drop)
- `VAL_SPLIT_2PCT`: Mol-Instruction train의 2%를 val로 분할 (Stage A→B에서만 발생)
- `SMILES_FAIL`: Stage A→B에서 SMILES/graph/SELFIES 변환 실패 잔여분
- `EVAL_LEAKAGE + FAMILY_DEDUP`: Stage B→C
- `SELFIES_FAIL`: Stage C→D SELFIES strict 재필터
- `REPHRASE_10X`: Stage D→E 4 target task Train만 10× (증가)


In [15]:
try:
    from dataset_generation.dedup import ENTITY_FAMILIES, REMOVE_ON_CONFLICT
except Exception as e:
    print(f"[Warning] dedup import 실패 ({e}); diagnostics에서 family 정보 비움")
    ENTITY_FAMILIES, REMOVE_ON_CONFLICT = {}, {}

MOL_INST_TASKS = set(SOURCE_TASK_MAP["Mol-Instruction"])
NAME_CONV_TASKS = {"smol-name_conversion-i2s", "smol-name_conversion-s2i"}
TARGET_10X = {"chebi-20-mol2text", "chebi-20-text2mol",
              "smol-molecule_captioning", "smol-molecule_generation"}


def explain_ab(task, split, a, b):
    """Stage A → B drop 원인 설명"""
    drop = a - b
    reasons = []
    if task in NAME_CONV_TASKS and split in ("val", "test"):
        return f"NAME_CONV_SKIP (val/test 생성 안 함; drop={drop:,})"
    if task in MOL_INST_TASKS and split == "val" and a == 0 and b > 0:
        return f"VAL_SPLIT_2PCT (train의 2%를 val로; +{b:,})"
    if task in MOL_INST_TASKS and split == "train":
        expected_val = int(a * 0.02)
        remainder = drop - expected_val
        reasons.append(f"VAL_SPLIT_2PCT (~{expected_val:,})")
        if abs(remainder) > 10:
            reasons.append(f"SMILES_FAIL (~{remainder:,})")
    elif drop > 0:
        reasons.append(f"SMILES_FAIL (~{drop:,})")
    elif drop < 0:
        reasons.append(f"(B > A, count 기준 불일치: {-drop:,})")
    return " + ".join(reasons) if reasons else "—"


def explain_bc(task, split, b, c):
    drop = b - c
    if drop <= 0:
        return "—"
    family = ENTITY_FAMILIES.get(task)
    role = None
    if family:
        if REMOVE_ON_CONFLICT.get(family) == task:
            role = f"REMOVE in {family}"
        else:
            role = f"ANCHOR in {family}"
    if split == "train":
        base = "EVAL_LEAKAGE"
        if family and REMOVE_ON_CONFLICT.get(family) == task:
            base += " + FAMILY_DEDUP"
        return f"{base} ({role or '-'}; drop={drop:,})"
    return f"(unexpected drop in {split}; {drop:,})"


def explain_cd(task, split, c, d):
    drop = c - d
    if drop <= 0:
        return "—"
    return f"SELFIES_FAIL (strict 재필터; drop={drop:,})"


def explain_de(task, split, d, e):
    delta = e - d
    if task in TARGET_10X and split == "train":
        ratio = e / d if d else 0
        return f"REPHRASE_10X ({ratio:.1f}x; +{delta:,})"
    if delta == 0:
        return "—"
    return f"(unexpected delta; {delta:+,})"


diag_rows = []
for _, row in df_master.iterrows():
    for split in ["train", "val", "test"]:
        a, b = row[f"A_{split}"], row[f"B_{split}"]
        c, d, e = row[f"C_{split}"], row[f"D_{split}"], row[f"E_{split}"]
        diag_rows.append({
            "Task": row["Task"], "Split": split,
            "A": a, "→B": b, "(A→B cause)": explain_ab(row["Task"], split, a, b),
            "→C": c, "(B→C cause)": explain_bc(row["Task"], split, b, c),
            "→D": d, "(C→D cause)": explain_cd(row["Task"], split, c, d),
            "→E": e, "(D→E cause)": explain_de(row["Task"], split, d, e),
        })

df_diag = pd.DataFrame(diag_rows)
display(df_diag.style.format({c: "{:,}" for c in ["A", "→B", "→C", "→D", "→E"]}))


,Task,Split,A,→B,(A→B cause),→C,(B→C cause),→D,(C→D cause),→E,(D→E cause)
0,chebi-20-mol2text,train,"26,402","26,402",—,"25,908",EVAL_LEAKAGE (ANCHOR in MOL2TEXT_FAMILY; drop=494),"25,908",—,"259,080","REPHRASE_10X (10.0x; +233,172)"
1,chebi-20-mol2text,val,"3,299","3,299",—,"3,299",—,"3,299",—,"3,299",—
2,chebi-20-mol2text,test,"3,297","3,297",—,"3,297",—,"3,297",—,"3,297",—
3,chebi-20-text2mol,train,"26,402","26,402",—,"25,908",EVAL_LEAKAGE (ANCHOR in TEXT2MOL_FAMILY; drop=494),"25,908",—,"259,080","REPHRASE_10X (10.0x; +233,172)"
4,chebi-20-text2mol,val,"3,299","3,299",—,"3,299",—,"3,299",—,"3,299",—
5,chebi-20-text2mol,test,"3,297","3,297",—,"3,297",—,"3,297",—,"3,297",—
6,forward_reaction_prediction,train,"124,384","121,896","VAL_SPLIT_2PCT (~2,487)","28,107","EVAL_LEAKAGE + FAMILY_DEDUP (REMOVE in REACTION_FORWARD_FAMILY; drop=93,789)","28,104",SELFIES_FAIL (strict 재필터; drop=3),"28,104",—
7,forward_reaction_prediction,val,0,"2,488","VAL_SPLIT_2PCT (train의 2%를 val로; +2,488)","2,488",—,"2,488",—,"2,488",—
8,forward_reaction_prediction,test,"1,000","1,000",—,"1,000",—,"1,000",—,"1,000",—
9,qm9_homo,train,"120,062","117,660","VAL_SPLIT_2PCT (~2,401)","117,656",EVAL_LEAKAGE (-; drop=4),"117,656",—,"117,656",—


## Section 8 — Save snapshots (optional)

필요 시 CSV로 저장.


In [16]:
SAVE_CSVS = False
if SAVE_CSVS:
    out_dir = os.path.join(PROJECT_ROOT, "dataset", "summary_reports")
    os.makedirs(out_dir, exist_ok=True)
    df_master.to_csv(os.path.join(out_dir, "pipeline_master.csv"), index=False)
    df_group.to_csv(os.path.join(out_dir, "pipeline_group_summary.csv"), index=False)
    df_diag.to_csv(os.path.join(out_dir, "pipeline_diagnostics.csv"), index=False)
    print(f"Saved to {out_dir}")
else:
    print("SAVE_CSVS=False (skip). True로 바꾸면 dataset/summary_reports/에 CSV 저장.")


SAVE_CSVS=False (skip). True로 바꾸면 dataset/summary_reports/에 CSV 저장.
